In [9]:
# TASK 8: Train Models for 10-20 Epochs
print("="*60)
print("TASK 8: TRAINING ALL MODELS")
print("="*60)

# Training configuration - FIXED HYPERPARAMETERS as per guide requirements
EPOCHS = 15
BATCH_SIZE = 32
VERBOSE = 1
FIXED_LEARNING_RATE = 0.001  # Fixed learning rate as required
USE_CALLBACKS = False  # Set to False to meet guide requirements for fixed hyperparameters

# Store training histories
training_histories = {}
trained_models = {}

print(f"Training configuration (FIXED HYPERPARAMETERS):")
print(f"  • Epochs: {EPOCHS} (fixed)")
print(f"  • Batch size: {BATCH_SIZE} (fixed)")
print(f"  • Learning rate: {FIXED_LEARNING_RATE} (fixed - no reduction)")
print(f"  • Optimizer: Adam (fixed)")
print(f"  • Callbacks enabled: {USE_CALLBACKS}")

if USE_CALLBACKS:
    print("  ⚠️  Using callbacks - this may modify hyperparameters during training")
    # Early stopping callback to prevent overfitting
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=0
    )

    # Reduce learning rate callback
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=0
    )

    callbacks = [early_stopping, reduce_lr]
    print(f"  • Early stopping: patience=5")
    print(f"  • Learning rate reduction: factor=0.5, patience=3")
else:
    callbacks = []
    print("  ✅ No callbacks - hyperparameters remain fixed throughout training")

print(f"\n{'='*60}")
print("STARTING TRAINING WITH FIXED HYPERPARAMETERS")
print(f"{'='*60}")

# Train all models
for model_name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Training: {model_name.upper()}")
    print(f"{'='*40}")

    # Ensure model uses fixed learning rate (recompile to be sure)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=FIXED_LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    print(f"📋 Model compiled with fixed LR: {FIXED_LEARNING_RATE}")

    # Train the model with fixed hyperparameters
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,  # Empty list if USE_CALLBACKS=False
        verbose=VERBOSE
    )

    # Store results
    training_histories[model_name] = history
    trained_models[model_name] = model

    # Display final metrics
    final_epoch = len(history.history['loss'])
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]

    print(f"\n📊 Final Results for {model_name}:")
    print(f"  Training completed at epoch: {final_epoch}")
    print(f"  Train accuracy: {final_train_acc:.4f}")
    print(f"  Validation accuracy: {final_val_acc:.4f}")
    print(f"  Train loss: {final_train_loss:.4f}")
    print(f"  Validation loss: {final_val_loss:.4f}")
    print(f"  Overfitting gap: {abs(final_train_acc - final_val_acc):.4f}")

    # Verify learning rate remained fixed
    current_lr = model.optimizer.learning_rate.numpy()
    print(f"  Final learning rate: {current_lr} (should be {FIXED_LEARNING_RATE})")

print(f"\n🎉 All {len(models)} models trained successfully!")
print("✅ Task 8 completed: Model training finished with FIXED HYPERPARAMETERS")
print(f"✅ Compliance: Fixed LR ({FIXED_LEARNING_RATE}), Fixed optimizer (Adam), Fixed epochs ({EPOCHS})")

TASK 8: TRAINING ALL MODELS
Training configuration (FIXED HYPERPARAMETERS):
  • Epochs: 15 (fixed)
  • Batch size: 32 (fixed)
  • Learning rate: 0.001 (fixed - no reduction)
  • Optimizer: Adam (fixed)
  • Callbacks enabled: False
  ✅ No callbacks - hyperparameters remain fixed throughout training

STARTING TRAINING WITH FIXED HYPERPARAMETERS

Training: BASIC_RELU
📋 Model compiled with fixed LR: 0.001
Epoch 1/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 120ms/step - accuracy: 0.5102 - loss: 0.6871 - precision: 0.3499 - recall: 0.3831 - val_accuracy: 0.7191 - val_loss: 0.5749 - val_precision: 0.8750 - val_recall: 0.3088
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7529 - loss: 0.5474 - precision: 0.8374 - recall: 0.4018 - val_accuracy: 0.7135 - val_loss: 0.5133 - val_precision: 0.6491 - val_recall: 0.5441
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7658 - loss: 0.4837 - precision: 0.7544 - recall: 0.5825 - val_accuracy: 0.7753 - val_loss: 0.4598 - val_precisio

In [10]:
# TASK 9: Evaluation Metrics
print("="*60)
print("TASK 9: MODEL EVALUATION METRICS")
print("="*60)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score

def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive evaluation of a trained model
    """
    # Make predictions
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    y_pred_proba = y_pred_proba.flatten()

    # Calculate metrics
    test_loss, test_accuracy, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)

    # Create results dictionary
    results = {
        'model_name': model_name,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'roc_auc': roc_auc,
        'f1_score': f1,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }

    return results

# Evaluate all trained models
evaluation_results = {}

print("Evaluating all models on test set...")
print(f"Test set size: {len(X_test)} samples")

for model_name, model in trained_models.items():
    print(f"\n--- Evaluating {model_name.upper()} ---")

    results = evaluate_model(model, X_test, y_test, model_name)
    evaluation_results[model_name] = results

    print(f"Test Accuracy: {results['test_accuracy']:.4f}")
    print(f"Test Precision: {results['test_precision']:.4f}")
    print(f"Test Recall: {results['test_recall']:.4f}")
    print(f"F1-Score: {results['f1_score']:.4f}")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")

# Create comprehensive results comparison
print(f"\n{'='*80}")
print("COMPREHENSIVE MODEL COMPARISON")
print(f"{'='*80}")

# Create comparison DataFrame
comparison_data = []
for model_name, results in evaluation_results.items():
    comparison_data.append({
        'Model': model_name,
        'Architecture': 'Advanced' if 'advanced' in model_name else 'Basic',
        'Activation': model_name.split('_')[-1].upper(),
        'Test_Accuracy': results['test_accuracy'],
        'Test_Precision': results['test_precision'],
        'Test_Recall': results['test_recall'],
        'F1_Score': results['f1_score'],
        'ROC_AUC': results['roc_auc'],
        'Test_Loss': results['test_loss']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test_Accuracy', ascending=False)

print("\nTop 3 Best Performing Models (by Test Accuracy):")
print(comparison_df.head(3)[['Model', 'Test_Accuracy', 'F1_Score', 'ROC_AUC']].to_string(index=False))

print("\nArchitecture Comparison:")
arch_comparison = comparison_df.groupby('Architecture')[['Test_Accuracy', 'F1_Score', 'ROC_AUC']].mean()
print(arch_comparison)

print("\nActivation Function Comparison:")
activation_comparison = comparison_df.groupby('Activation')[['Test_Accuracy', 'F1_Score', 'ROC_AUC']].mean()
print(activation_comparison)

# Detailed classification report for best model
best_model_name = comparison_df.iloc[0]['Model']
best_results = evaluation_results[best_model_name]

print(f"\n{'='*50}")
print(f"DETAILED REPORT - BEST MODEL: {best_model_name.upper()}")
print(f"{'='*50}")

print("\nClassification Report:")
print(classification_report(y_test, best_results['y_pred'],
                          target_names=['Died', 'Survived']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, best_results['y_pred'])
print(f"""
True Negatives (Correctly predicted died): {cm[0,0]}
False Positives (Incorrectly predicted survived): {cm[0,1]}
False Negatives (Incorrectly predicted died): {cm[1,0]}
True Positives (Correctly predicted survived): {cm[1,1]}
""")

print("✅ Task 9 completed: Comprehensive model evaluation finished")

TASK 9: MODEL EVALUATION METRICS
Evaluating all models on test set...
Test set size: 179 samples

--- Evaluating BASIC_RELU ---
Test Accuracy: 0.7989
Test Precision: 0.8511
Test Recall: 0.5797
F1-Score: 0.6897
ROC-AUC: 0.8470

--- Evaluating ADVANCED_RELU ---
Test Accuracy: 0.6704
Test Precision: 0.9167
Test Recall: 0.1594
F1-Score: 0.2716
ROC-AUC: 0.8315

--- Evaluating BASIC_LEAKY_RELU ---


Test Accuracy: 0.7933
Test Precision: 0.7424
Test Recall: 0.7101
F1-Score: 0.7259
ROC-AUC: 0.8358

--- Evaluating ADVANCED_LEAKY_RELU ---


Test Accuracy: 0.7877
Test Precision: 0.8974
Test Recall: 0.5072
F1-Score: 0.6481
ROC-AUC: 0.8382

--- Evaluating BASIC_GELU ---
Test Accuracy: 0.7821
Test Precision: 0.6974
Test Recall: 0.7681
F1-Score: 0.7310
ROC-AUC: 0.8373

--- Evaluating ADVANCED_GELU ---
Test Accuracy: 0.6927
Test Precision: 0.9375
Test Recall: 0.2174
F1-Score: 0.3529
ROC-AUC: 0.8258

COMPREHENSIVE MODEL COMPARISON

Top 3 Best Performing Models (by Test Accuracy):
              Model  Test_Accuracy  F1_Score  ROC_AUC
         basic_relu       0.798883  0.689655 0.847036
   basic_leaky_relu       0.793296  0.725926 0.835837
advanced_leaky_relu       0.787709  0.648148 0.838208

Architecture Comparison:
              Test_Accuracy  F1_Score   ROC_AUC
Architecture                                   
Advanced           0.716946  0.424231  0.831840
Basic              0.791434  0.715539  0.840053

Activation Function Comparison:
            Test_Accuracy  F1_Score   ROC_AUC
Activation                                   
